# Week 30, Day 5: Many channels, one behaviour: the channel zoo and degeneracy

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w030/d5.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Hundreds of neurons at once

`rate_of(gNa, gK, gL)` simulates one HH model per entry of the arrays for 130 ms at 10 μA/cm² and returns each model's firing rate after the first 30 ms. Draw 300 models with gNa uniform on 60 to 240, gK on 18 to 72 and gL on 0.15 to 0.6 (local rng, seed 30), compute their rates `f` and the original's rate `f_hh`, and mark as `good` those within 3 Hz of the original.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
ENa, EK, EL = 50.0, -77.0, -54.387

def rates(V):
    V = V + 1e-7
    an = 0.01 * (V + 55) / (1 - np.exp(-(V + 55) / 10)); bn = 0.125 * np.exp(-(V + 65) / 80)
    am = 0.1 * (V + 40) / (1 - np.exp(-(V + 40) / 10)); bm = 4.0 * np.exp(-(V + 65) / 18)
    ah = 0.07 * np.exp(-(V + 65) / 20); bh = 1 / (1 + np.exp(-(V + 35) / 10))
    return an, bn, am, bm, ah, bh

def rate_of(gNa, gK, gL, I=10.0, T=130.0, dt=0.01):
    k = gNa.size
    V = np.full(k, -65.0); n = np.full(k, 0.3177); m = np.full(k, 0.0529); h = np.full(k, 0.5961)
    first = np.full(k, np.nan); last = np.full(k, np.nan); count = np.zeros(k)
    for i in range(int(T / dt)):
        an, bn, am, bm, ah, bh = rates(V)
        dV = I - gNa * m**3 * h * (V - ENa) - gK * n**4 * (V - EK) - gL * (V - EL)
        n += dt * (an * (1 - n) - bn * n); m += dt * (am * (1 - m) - bm * m); h += dt * (ah * (1 - h) - bh * h)
        Vnew = V + dt * dV
        if i * dt >= 30:
            up = (Vnew > 0) & (V <= 0)
            first = np.where(up & np.isnan(first), i * dt, first); last = np.where(up, i * dt, last); count += up
        V = Vnew
    span = np.where(count >= 2, last - first, 1.0)
    return np.where(count >= 2, 1000 * (count - 1) / span, 0.0)

r = np.random.default_rng(30)
N = 300
gNa = r.uniform(60, 240, N); gK = r.uniform(18, 72, N); gL = r.uniform(0.15, 0.6, N)
f = ...
f_hh = rate_of(np.array([120.0]), np.array([36.0]), np.array([0.3]))[0]
good = ...
print(f"original HH: {f_hh:.1f} Hz;  {np.sum(f == 0)} silent models, {good.sum()} of {N} within 3 Hz of the original")
print(f"good models: gNa from {gNa[good].min():.0f} to {gNa[good].max():.0f}, gK from {gK[good].min():.0f} to {gK[good].max():.0f}, gL from {gL[good].min():.2f} to {gL[good].max():.2f}")
plt.scatter(gNa, gK, c="lightgray", s=10); plt.scatter(gNa[good], gK[good], c="C3", s=14)
plt.scatter([120], [36], marker="*", s=150, c="k"); plt.xlabel("gNa (mS/cm^2)"); plt.ylabel("gK (mS/cm^2)"); plt.show()

In [ ]:
import numpy as np
def test_rates():
    "the original HH parameters fire at about 68 Hz, and some random models are silent"
    assert 66 < f_hh < 71 and np.sum(f == 0) > 0

def test_good():
    "good marks models within 3 Hz of the original rate, and there are more than 20 of them"
    assert np.array_equal(good, np.abs(f - f_hh) <= 3) and good.sum() > 20

test_rates()
test_good()
print("All checks passed")

### Exercise: Correlated conductances

Among the good models, compute the correlation `rho` between gNa and gK, and the coefficient of variation (standard deviation over mean) of the ratio gK / gNa among good models (`cv_good`) and among all models (`cv_all`). Then simulate the model whose conductances are the averages of the good models'.

In [ ]:
rho = ...
ratio_all, ratio_good = gK / gNa, gK[good] / gNa[good]
cv_all = ...
cv_good = ...
f_avg = rate_of(np.array([gNa[good].mean()]), np.array([gK[good].mean()]), np.array([gL[good].mean()]))[0]
print(f"correlation of gNa and gK among good models: {rho:.2f}")
print(f"variation of gK/gNa: {cv_good:.2f} among good models, {cv_all:.2f} among all")
print(f"the average good model fires at {f_avg:.1f} Hz (target {f_hh:.1f})")

In [ ]:
import numpy as np
def test_corr():
    "rho is the correlation of gNa and gK among good models, and exceeds 0.8"
    assert abs(rho - np.corrcoef(gNa[good], gK[good])[0, 1]) < 1e-12 and rho > 0.8

def test_cv():
    "the ratio gK / gNa varies much less among good models than among all models"
    assert cv_good < 0.6 * cv_all

test_corr()
test_cv()
print("All checks passed")